In [1]:
import os, gc, warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split

# list input files (debug)
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))



2026-05-02 21:41:44.003697: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 21:41:44.015276: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777758104.029028      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777758104.032601      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 21:41:44.046846: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/kaggle/input/train.tsv.zip
/kaggle/input/sampleSubmission.csv
/kaggle/input/test.tsv.zip
/kaggle/input/description.md
/kaggle/input/sampleSubmission.csv.zip
/kaggle/input/test.tsv
/kaggle/input/train.tsv
/kaggle/input/movie-review-sentiment-analysis-kernels-only/train.tsv.zip
/kaggle/input/movie-review-sentiment-analysis-kernels-only/sampleSubmission.csv
/kaggle/input/movie-review-sentiment-analysis-kernels-only/test.tsv.zip
/kaggle/input/movie-review-sentiment-analysis-kernels-only/description.md
/kaggle/input/movie-review-sentiment-analysis-kernels-only/sampleSubmission.csv.zip
/kaggle/input/movie-review-sentiment-analysis-kernels-only/test.tsv
/kaggle/input/movie-review-sentiment-analysis-kernels-only/train.tsv


In [2]:
train_path = "../input/movie-review-sentiment-analysis-kernels-only/train.tsv.zip"
test_path = "../input/movie-review-sentiment-analysis-kernels-only/test.tsv.zip"
sub_path = "../input/movie-review-sentiment-analysis-kernels-only/sampleSubmission.csv"

train = pd.read_csv(train_path, sep="\t", encoding="UTF-8")
test = pd.read_csv(test_path, sep="\t", encoding="UTF-8")
sub = pd.read_csv(sub_path)
print(train.shape, test.shape, sub.shape)



(109242, 4) (46818, 3) (46818, 2)


In [3]:
seed = 101
np.random.seed(seed)
tf.random.set_seed(seed)

X = train["Phrase"].astype(str).values
y = to_categorical(train["Sentiment"])
num_classes = y.shape[1]
print("Number of classes:", num_classes)



Number of classes: 5


In [4]:
max_features = 15000
maxlen = 50

tokenizer = Tokenizer(num_words=max_features, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)
X_seq = pad_sequences(X_seq, maxlen=maxlen, padding="post", truncating="post")

test_seq = tokenizer.texts_to_sequences(test["Phrase"].astype(str).values)
test_seq = pad_sequences(test_seq, maxlen=maxlen, padding="post", truncating="post")

X_train, X_val, y_train, y_val = train_test_split(
    X_seq, y, test_size=0.2, stratify=y, random_state=seed
)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)



(87393, 50) (21849, 50) (87393, 5) (21849, 5)


In [5]:
embed_dim = 100  # size of embedding vectors
# Random embedding matrix (since pretrained file is unavailable)
embedding_matrix = np.random.normal(
    loc=0.0, scale=1.0, size=(max_features, embed_dim)
).astype(np.float32)


def get_model(max_features, embed_dim, embedding_matrix):
    K.clear_session()
    model = Sequential()
    model.add(
        Embedding(
            input_dim=max_features,
            output_dim=embed_dim,
            weights=[embedding_matrix],
            input_length=maxlen,
            trainable=True,
        )
    )
    model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
    model.add(Dense(num_classes, activation="softmax"))
    model.compile(
        loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
    )
    model.summary()
    return model


model = get_model(max_features, embed_dim, embedding_matrix)



2026-05-02 21:41:54.144047: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     1,500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,500,000 (5.72 MB)

 Trainable params: 1,500,000 (5.72 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
batch_size = 128
epochs = 10

model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    verbose=2,
)



Epoch 1/10
683/683 - 26s - 38ms/step - accuracy: 0.5084 - loss: 1.2688 - val_accuracy: 0.5086 - val_loss: 1.2061
Epoch 2/10
683/683 - 24s - 34ms/step - accuracy: 0.5208 - loss: 1.1767 - val_accuracy: 0.5381 - val_loss: 1.1442
Epoch 3/10
683/683 - 23s - 33ms/step - accuracy: 0.5539 - loss: 1.1013 - val_accuracy: 0.5732 - val_loss: 1.0704
Epoch 4/10
683/683 - 23s - 33ms/step - accuracy: 0.6018 - loss: 1.0004 - val_accuracy: 0.6079 - val_loss: 0.9813
Epoch 5/10
683/683 - 23s - 34ms/step - accuracy: 0.6359 - loss: 0.9148 - val_accuracy: 0.6227 - val_loss: 0.9397
Epoch 6/10
683/683 - 24s - 35ms/step - accuracy: 0.6574 - loss: 0.8593 - val_accuracy: 0.6301 - val_loss: 0.9115
Epoch 7/10
683/683 - 24s - 35ms/step - accuracy: 0.6713 - loss: 0.8196 - val_accuracy: 0.6350 - val_loss: 0.8995
Epoch 8/10
683/683 - 25s - 36ms/step - accuracy: 0.6833 - loss: 0.7881 - val_accuracy: 0.6397 - val_loss: 0.8834
Epoch 9/10
683/683 - 25s - 37ms/step - accuracy: 0.6947 - loss: 0.7626 - val_accuracy: 0.6426 - 

In [7]:
# Predict classes for the test set
pred_probs = model.predict(test_seq, batch_size=batch_size, verbose=0)
pred_classes = np.argmax(pred_probs, axis=1)

sub["Sentiment"] = pred_classes
submission_path = "submission.csv"
sub.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")
print(sub.head())

Submission saved to submission.csv
   PhraseId  Sentiment
0     21924          2
1     18207          3
2     81803          2
3    114983          1
4    135928          4
